# Comparison Across Training Seeds

This notebook compares final-evaluation results across training seeds `156`, `157`, and `158` for each task, using experiments `001–003`.

The related unseen split (`test_unseen_related`) is excluded for object and object-region classification because these results are not scored. `validation_unseen_matched` is included for reference and is not an independent test result.

It exports one CSV summary table per available task, containing individual seed values and summary statistics. Tasks without an `experiments.parquet` file are skipped.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [2]:
RESULTS_DIR = Path("../results/final_evaluation")
OUTPUT_DIR = Path("../exports/seed_comparison")
EXPERIMENT_NUMBERS = [1, 2, 3]
SEEDS = [156, 157, 158]

EVALUATION_SPLITS = [
    "test",
    "test_unseen_matched",
    "test_unseen_related",
    "validation_unseen_matched",
]

TASK_METRICS = {
    "object_classification": ["accuracy", "macro_f1_avg"],
    "object_region_classification": ["accuracy", "macro_f1_avg"],
    "force_level_classification": ["accuracy", "macro_f1_avg"],
    "object_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "object_region_conditioned_classification": ["accuracy", "macro_f1_avg"],
    "force_n_regression": ["mae", "rmse", "r2"],
    "fsr_voltage_regression": ["mae", "rmse", "r2"],
}

UNSCORED_RELATED_TASKS = {
    "object_classification",
    "object_region_classification",
    "object_conditioned_classification",
    "object_region_conditioned_classification",
}

In [3]:
def load_seed_results(task):
    """
    Load one final-evaluation run per experiment and training seed.

    Args:
        task (str): The task folder under the final-evaluation results directory.

    Returns:
        pd.DataFrame: The selected result rows indexed in training-seed order.
    """

    results_path = RESULTS_DIR / task / "experiments.parquet"
    results = pd.read_parquet(results_path)
    results = results.loc[results["experiment_number"].isin(EXPERIMENT_NUMBERS)].copy()

    # Keep seed values and summary columns in the same order across all task tables
    return results.set_index("seed").loc[SEEDS]


def calculate_seed_summary(task, results):
    """
    Calculate per-split metric statistics across the three training seeds.

    Args:
        task (str): The task folder identifying the metrics and scored splits.
        results (pd.DataFrame): The selected result rows indexed by training seed.

    Returns:
        pd.DataFrame: Individual seed values and summary statistics per metric.
    """

    rows = []
    for split in EVALUATION_SPLITS:
        # Skip unscored classification results while retaining force-task metrics
        if split == "test_unseen_related" and task in UNSCORED_RELATED_TASKS:
            continue

        for metric in TASK_METRICS[task]:
            column = f"{split}_{metric}"
            values = results.loc[SEEDS, column]
            row = {"split": split, "metric": metric}
            row.update({f"seed_{seed}": values.loc[seed] for seed in SEEDS})
            row.update(
                {
                    "n": values.count(),
                    "mean": values.mean(),
                    "std": values.std(ddof=1),
                    "min": values.min(),
                    "max": values.max(),
                }
            )
            rows.append(row)

    return pd.DataFrame(rows)


def export_seed_comparisons():
    """
    Export a seed-comparison CSV table for each available final-evaluation task.

    Tasks without an experiments.parquet file are skipped.

    Returns:
        dict[str, pd.DataFrame]: The summary table for each task.
    """

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    comparisons = {}
    for task in TASK_METRICS:
        if not (RESULTS_DIR / task / "experiments.parquet").is_file():
            print(f"Skipping {task}: experiments.parquet not found.")
            continue

        results = load_seed_results(task)
        comparison = calculate_seed_summary(task, results)
        comparison.to_csv(OUTPUT_DIR / f"{task}.csv", index=False)
        comparisons[task] = comparison

    return comparisons

In [4]:
# Run available task comparisons and retain the tables for display below
seed_comparisons = export_seed_comparisons()

Skipping force_level_classification: experiments.parquet not found.
Skipping object_conditioned_classification: experiments.parquet not found.
Skipping object_region_conditioned_classification: experiments.parquet not found.
Skipping force_n_regression: experiments.parquet not found.
Skipping fsr_voltage_regression: experiments.parquet not found.


In [5]:
# Display each task table
with pd.option_context("display.max_rows", None, "display.precision", 4):
    for task, comparison in seed_comparisons.items():
        display(task)
        display(comparison)

'object_classification'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,accuracy,92.1327,91.3377,93.3498,3,92.2734,1.0134,91.3377,93.3498
1,test,macro_f1_avg,0.9233,0.9149,0.9346,3,0.9243,0.0098,0.9149,0.9346
2,test_unseen_matched,accuracy,47.7469,50.4321,52.1296,3,50.1029,2.2098,47.7469,52.1296
3,test_unseen_matched,macro_f1_avg,0.5990,0.5978,0.6297,3,0.6088,0.0181,0.5978,0.6297
4,validation_unseen_matched,accuracy,53.8194,50.2639,54.0000,3,52.6944,2.1069,50.2639,54.0000
5,validation_unseen_matched,macro_f1_avg,0.5757,0.5489,0.5771,3,0.5672,0.0159,0.5489,0.5771


'object_region_classification'

,split,metric,seed_156,seed_157,seed_158,n,mean,std,min,max
0,test,accuracy,88.3662,88.4375,88.4759,3,88.4265,0.0556,88.3662,88.4759
1,test,macro_f1_avg,0.8752,0.8740,0.8767,3,0.8753,0.0014,0.8740,0.8767
2,test_unseen_matched,accuracy,34.4753,36.2500,35.5093,3,35.4115,0.8914,34.4753,36.2500
3,test_unseen_matched,macro_f1_avg,0.4220,0.4547,0.4338,3,0.4369,0.0166,0.4220,0.4547
4,validation_unseen_matched,accuracy,42.2083,41.8750,40.0000,3,41.3611,1.1905,40.0000,42.2083
5,validation_unseen_matched,macro_f1_avg,0.4622,0.4655,0.4328,3,0.4535,0.0180,0.4328,0.4655
